# bio.tools pair, edge and node tables

Builds three files from the merged pair table:

| file | one row per | key columns |
|---|---|---|
| `pairs_biotools.csv` | tool pair (A to B) | everything about the pair, incl. `direction_type` |
| `biotools_edges.csv` | tool pair (A to B) | `source`, `target`, `direction_type`, `n_workflows`, `workflow_ids` |
| `biotools_pairs.csv` | workflow and tool pair | `workflow_id`, `source`, `target`, `direction_type`, names |
| `biotools_nodes.csv` | tool | `biotoolsID`, `biotools_name`, `n_workflows`, `matching_method` |

`direction_type`: `one_direction` (A feeds B, never B feeds A), `both_directions` (A feeds B and B feeds A in sequences), `parallel` (side by side, no path between them).

Run the cells from top to bottom. The inputs are read from `RESULTS_DIR` and the tables are written to `OUT_DIR`; both are set in the next cell.

In [ ]:
import os
import pandas as pd

# The notebook lives in nf-core/scripts/3_nodes_edges. Open it there, or change the two folders below.
RESULTS_DIR = "../../results"   # written by run.py: pairs_merged_unique.csv, tools_with_ids.csv; put the bio.tools dump here too
OUT_DIR = "../../data"          # the four tables are written here

TOOLS_FILE   = os.path.join(RESULTS_DIR, "tools_with_ids.csv")        # from: python run.py idtable
PAIRS_FILE   = os.path.join(RESULTS_DIR, "pairs_merged_unique.csv")   # from: python run.py merge (must contain direction_type)
BIOTOOLS_RAW = os.path.join(RESULTS_DIR, "biotools_raw_14-10-2026.jsonl")   # bio.tools dump (not in git), used only for node names

os.makedirs(OUT_DIR, exist_ok=True)
for f in (TOOLS_FILE, PAIRS_FILE, BIOTOOLS_RAW):
    print(("found    " if os.path.exists(f) else "MISSING  ") + os.path.abspath(f))

## 1. Tools that have a bio.tools ID

In [ ]:
tools = pd.read_csv(TOOLS_FILE)
has_id = tools["biotools_id"].notna() & tools["biotools_id"].astype(str).str.strip().ne("")
biotools_software = set(tools.loc[has_id, "tool"].dropna().astype(str).str.strip())

print(f"Tools in the table:            {len(tools):,}")
print(f"  with a bio.tools ID:         {has_id.sum():,}")
print(f"  without a bio.tools ID:      {(~has_id).sum():,}")

## 2. Pairs where both tools have a bio.tools ID and at least one positive workflow

In [ ]:
pairs = pd.read_csv(PAIRS_FILE)
assert "direction_type" in pairs.columns, "no direction_type column: re-run the merge step with the updated code"

both_ids = (pairs["source_software"].astype(str).str.strip().isin(biotools_software)
            & pairs["target_software"].astype(str).str.strip().isin(biotools_software))
positive = pairs["n_pipelines_positive"] > 0

pairs_final = pairs.loc[both_ids & positive, [
    "source_software", "target_software",
    "source_biotools_id", "target_biotools_id",
    "source_match_method", "target_match_method",
    "direction_type",
    "n_pipelines_positive", "n_pipelines_negative", "n_process_edges_positive",
    "reverse_direction_pipelines", "pipelines_positive", "pipelines_negative",
]].rename(columns={
    "source_software": "source", "target_software": "target",
    "source_biotools_id": "source_biotool_id", "target_biotools_id": "target_biotool_id",
    "source_match_method": "source_match", "target_match_method": "target_match",
}).reset_index(drop=True)

# the merge step calls an ID taken from the module's meta.yml "meta_id"
pairs_final[["source_match", "target_match"]] = pairs_final[["source_match", "target_match"]].replace("meta_id", "biotool_id")

print(f"All pairs:                          {len(pairs):,}")
print(f"Both tools have a bio.tools ID:     {both_ids.sum():,}")
print(f"... and at least one positive:      {len(pairs_final):,}")
print()
print(pairs_final["direction_type"].value_counts().to_string())
pairs_final.head()

In [ ]:
pairs_final.to_csv(os.path.join(OUT_DIR, "pairs_biotools.csv"), index=False)
print("Saved pairs_biotools.csv in", os.path.abspath(OUT_DIR))

## 3. Edges: one row per tool pair

`source` and `target` are bio.tools IDs.

In [ ]:
edges = pairs_final[["source_biotool_id", "target_biotool_id", "direction_type",
                     "n_pipelines_positive", "pipelines_positive"]].rename(columns={
    "source_biotool_id": "source", "target_biotool_id": "target",
    "n_pipelines_positive": "n_workflows", "pipelines_positive": "workflow_ids",
})
edges.head()

## 4. Workflow pairs: one row per workflow and tool pair

`source_biotools_name` and `target_biotools_name` are the tool names used in the pipelines.

In [ ]:
workflow_pairs = (
    pairs_final.assign(workflow_id=pairs_final["pipelines_positive"].str.split(";"))
    .explode("workflow_id")
    [["workflow_id", "source_biotool_id", "target_biotool_id", "direction_type", "source", "target"]]
    .rename(columns={"source_biotool_id": "source", "target_biotool_id": "target",
                     "source": "source_biotools_name", "target": "target_biotools_name"})
    .reset_index(drop=True)
)
print(f"{len(workflow_pairs):,} workflow-pair rows")
workflow_pairs.head()

## 5. Nodes: one row per tool

In [ ]:
ends = pd.concat([
    pairs_final[["source_biotool_id", "source_match", "pipelines_positive"]]
        .set_axis(["biotoolsID", "matching_method", "pipelines_positive"], axis=1),
    pairs_final[["target_biotool_id", "target_match", "pipelines_positive"]]
        .set_axis(["biotoolsID", "matching_method", "pipelines_positive"], axis=1),
])
ends["workflow_id"] = ends["pipelines_positive"].str.split(";")
ends = ends.explode("workflow_id")

nodes = (ends.groupby("biotoolsID")
         .agg(n_workflows=("workflow_id", "nunique"), matching_method=("matching_method", "first"))
         .reset_index())
print(f"{len(nodes):,} nodes")
print(nodes["matching_method"].value_counts().to_string())

### Names from the bio.tools dump

In [ ]:
raw = pd.read_json(BIOTOOLS_RAW, lines=True)
names = (raw[["biotoolsID", "name"]].dropna()
         .assign(key=lambda d: d["biotoolsID"].astype(str).str.strip().str.casefold())
         .drop_duplicates("key").set_index("key")["name"])

nodes["biotools_name"] = nodes["biotoolsID"].astype(str).str.strip().str.casefold().map(names)
nodes = nodes[["biotoolsID", "biotools_name", "n_workflows", "matching_method"]]

print(f"Names found:   {nodes['biotools_name'].notna().sum():,}")
print(f"Names missing: {nodes['biotools_name'].isna().sum():,}")
nodes[nodes["biotools_name"].isna()]

## 6. Checks and save

In [ ]:
assert len(edges) == len(pairs_final)
assert len(workflow_pairs) == pairs_final["n_pipelines_positive"].sum()
assert set(nodes["biotoolsID"]) == set(edges["source"]) | set(edges["target"])
assert edges["direction_type"].notna().all() and workflow_pairs["direction_type"].notna().all()

nodes.to_csv(os.path.join(OUT_DIR, "biotools_nodes.csv"), index=False)
edges.to_csv(os.path.join(OUT_DIR, "biotools_edges.csv"), index=False)
workflow_pairs.to_csv(os.path.join(OUT_DIR, "biotools_pairs.csv"), index=False)
print("Saved biotools_nodes.csv, biotools_edges.csv, biotools_pairs.csv in", os.path.abspath(OUT_DIR))
print(f"nodes {len(nodes):,} | edges {len(edges):,} | workflow pairs {len(workflow_pairs):,}")